In [1]:
import os
import re

from datasets import load_dataset, Dataset, load_from_disk
from BPETokenizer import BPETokenizer as tk

/opt/homebrew/anaconda3/envs/buildinglargelanguagemodel/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Loading the Pre-Train Dataset

In [2]:
dataset_dict = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", streaming=True)

In [3]:
dataset = dataset_dict['train'].take(10000)

In [4]:
sample_batch = dataset.take(10)

for _, row in enumerate(sample_batch):
    print(row['text'][:500])

The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.
The freedom she exhibited in finally accepting him in direct defiance of Lady Cath
Taking Play Seriously
By ROBIN MARANTZ HENIG
Published: February 17, 2008
On a drizzly Tuesday night in late January, 200 people came out to hear a psychiatrist talk rhapsodically about play -- not just the intense, joyous play of children, but play for all people, at all ages, at all times. (All species too; the lecture featured touching photos of a polar bear and a husky engaging playfully at a snowy outpost in northern Canada.) Stuart Brown, president of the National Institute for Play, was 

In [5]:
local_data_list = list(dataset_dict['train'].take(10000))

In [6]:
local_dataset = Dataset.from_list(local_data_list)

In [7]:
if not os.path.exists("local_fineweb_data"):
    local_dataset.save_to_disk("local_fineweb_data/hf_format")

In [8]:
local_dataset = load_from_disk("local_fineweb_data/hf_format")

In [9]:
for index in range(10):
    print(local_dataset[index]['text'][:500])

The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.
The freedom she exhibited in finally accepting him in direct defiance of Lady Cath
Taking Play Seriously
By ROBIN MARANTZ HENIG
Published: February 17, 2008
On a drizzly Tuesday night in late January, 200 people came out to hear a psychiatrist talk rhapsodically about play -- not just the intense, joyous play of children, but play for all people, at all ages, at all times. (All species too; the lecture featured touching photos of a polar bear and a husky engaging playfully at a snowy outpost in northern Canada.) Stuart Brown, president of the National Institute for Play, was 

In [10]:
total_tokens = sum(local_dataset['token_count'])
print(f"the training data has {total_tokens} tokens")

the training data has 10302766 tokens


**REGEX PATTERN And Tokenization By Row**

In [11]:
REGEX_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?[a-zA-Z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+(?!\S)|\s+"""

### Building the Vocabulary

In [12]:
tokenizer = tk(REGEX_PATTERN, num_merges=1000)
tokenizer.build_vocab(local_dataset)
tokenized_with_ids = local_dataset.map(tokenizer.encode_row, num_proc=10)

Map (num_proc=10): 100%|██████████| 10000/10000 [01:35<00:00, 104.51 examples/s]
